<a href="https://colab.research.google.com/github/dhasan07/LSTM/blob/main/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import files
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Read the semi-colon delimited file from the uploaded archive
# Since it's a zip file containing the CSV, we can read it directly using pandas
df = pd.read_csv('archive (1).zip', sep=';')

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())

In [ ]:
# Clean column names from any surrounding quotes
df.columns = [c.replace('"', '').strip() for c in df.columns]
if 'district' in df.columns:
    df['district'] = df['district'].astype(str).str.replace('"', '').str.strip()
if 'state' in df.columns:
    df['state'] = df['state'].astype(str).str.replace('"', '').str.strip()

# Identify day columns (e.g., '1st', '2nd', ..., '31st'), explicitly excluding 'month'
day_cols = [
    c for c in df.columns
    if (c.endswith('st') or c.endswith('nd') or c.endswith('rd') or c.endswith('th'))
    and c != 'month'
]

# Melt columns representing days into rows
id_cols = [c for c in df.columns if c not in day_cols]
df_long = df.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="day_str",
    value_name="Rainfall"
)

# Extract numerical day using a raw string pattern to avoid syntax warnings
df_long['day'] = df_long['day_str'].str.extract(r'(\d+)').astype(float)

print("Long format data preview:")
display(df_long.head())
print("Unique columns in long format:", df_long.columns.tolist())

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Prepare the 'daily' DataFrame from 'df_long'
daily = df_long.copy()

# Since year is not present in the dataset, we'll assign a placeholder leap year (e.g., 2020) to handle February 29 safely.
daily['year'] = 2020

# Create a date string and parse with coerce to mark invalid dates (like Feb 30th or Nov 31st) as NaT
date_string = daily['year'].astype(str) + '-' + daily['month'].astype(str).str.zfill(2) + '-' + daily['day'].astype(int).astype(str).str.zfill(2)
daily['date'] = pd.to_datetime(date_string, format='%Y-%m-%d', errors='coerce')

# Filter out the invalid dates (where 'date' is NaT)
daily = daily.dropna(subset=['date']).sort_values(by=['state', 'district', 'date']).reset_index(drop=True)

# Apply scaling
scaler = MinMaxScaler()
daily["Rainfall_Normalized"] = scaler.fit_transform(daily[["Rainfall"]])

print("Daily DataFrame shape after filtering invalid dates:", daily.shape)
display(daily.head())

In [ ]:
def create_sequences(df, window_size=30):
    X, y = [], []
    # Group by state and district so sequences don't cross boundaries
    grouped = df.groupby(['state', 'district'])

    for (state, district), group in grouped:
        # Sort chronologically to maintain proper sequence order
        group_sorted = group.sort_values('date')
        values = group_sorted['Rainfall_Normalized'].values

        if len(values) > window_size:
            for i in range(len(values) - window_size):
                X.append(values[i : i + window_size])
                y.append(values[i + window_size])

    return np.array(X), np.array(y)

# Generate sequences (e.g., using past 30 days of rainfall to predict the next day)
window_size = 30
X, y = create_sequences(daily, window_size=window_size)

# Reshape X to fit LSTM expectations: [samples, time_steps, features]
X = X.reshape((X.shape[0], X.shape[1], 1))

print("Input sequences shape (X):", X.shape)
print("Target values shape (y):", y.shape)

In [ ]:
# Generate sequences using the requested previous 7 days window size
window_size = 7
X, y = create_sequences(daily, window_size=window_size)

# Reshape X to [samples, time_steps, features]
X = X.reshape((X.shape[0], X.shape[1], 1))

print("Input sequences shape (X) with 7-day window:", X.shape)
print("Target values shape (y):", y.shape)

In [ ]:
# Since we need a chronological training/testing split, we will use the chronological order of our data.
# Let's take the first 80% of chronological sequences for training and the remaining 20% for testing.
split_idx = int(len(X) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

In [ ]:
# Define the LSTM sequential model
model = Sequential([
    LSTM(50, activation='relu', input_shape=(window_size, 1), return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
# Train the model with early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# Plot the training and validation loss history
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', color='teal', linewidth=2)
if 'val_loss' in history.history:
    plt.plot(history.history['val_loss'], label='Validation Loss', color='orange', linewidth=2)

plt.title('LSTM Model Training & Validation Loss History', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)

# Calculate metrics on normalized scale
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Normalized MAE: {mae:.4f}")
print(f"Normalized RMSE: {rmse:.4f}")

# Inverse transform predictions and targets to original scale (mm of rainfall)
y_test_orig = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_orig = scaler.inverse_transform(y_pred.reshape(-1, 1))

mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)
rmse_orig = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))

print(f"Original Scale MAE: {mae_orig:.2f} mm")
print(f"Original Scale RMSE: {rmse_orig:.2f} mm")

### Model Performance Summary
Below are the evaluated model metrics (MAE and RMSE) on both the normalized scale and original scale (millimeters).

In [ ]:
print("=========================================")
print("     LSTM MODEL PERFORMANCE METRICS      ")
print("=========================================")
print(f"Normalized Scale:")
print(f"  - Mean Absolute Error (MAE):     {mae:.6f}")
print(f"  - Root Mean Squared Error (RMSE): {rmse:.6f}")
print("-----------------------------------------")
print(f"Original Scale (Millimeters):")
print(f"  - Mean Absolute Error (MAE):     {mae_orig:.4f} mm")
print(f"  - Root Mean Squared Error (RMSE): {rmse_orig:.4f} mm")
print("=========================================")

### Baseline Model Comparison

To see if our deep learning model brings value, we compare it against a simple **naive baseline**. This baseline predicts that next-day rainfall (Day 8) is simply equal to the rainfall of the most recent observed day (Day 7).

In [ ]:
# The last step of our input sequences corresponds to "today's" rainfall (Day 7)
# X_test has shape [samples, 7, 1] on the normalized scale
last_observed_normalized = X_test[:, -1, 0].reshape(-1, 1)

# Inverse transform the baseline prediction to original scale (mm)
y_baseline_orig = scaler.inverse_transform(last_observed_normalized)

# Calculate baseline evaluation metrics on the original scale
baseline_mae = mean_absolute_error(y_test_orig, y_baseline_orig)
baseline_rmse = np.sqrt(mean_squared_error(y_test_orig, y_baseline_orig))

# Display the comparison table
print("===========================================================")
print("         MODEL COMPARISON (ORIGINAL SCALE in mm)          ")
print("===========================================================")
print(f"Naive Baseline (Last Value Model):")
print(f"  - MAE:  {baseline_mae:.4f} mm")
print(f"  - RMSE: {baseline_rmse:.4f} mm")
print("-----------------------------------------------------------")
print(f"LSTM Neural Network:")
print(f"  - MAE:  {mae_orig:.4f} mm")
print(f"  - RMSE: {rmse_orig:.4f} mm")
print("===========================================================")

In [ ]:
# Plot actual vs. predicted rainfall
plt.figure(figsize=(12, 6))
plt.plot(y_test_orig[:200], label='Actual Rainfall', color='blue', alpha=0.7)
plt.plot(y_pred_orig[:200], label='Predicted Rainfall', color='orange', alpha=0.9)
plt.title('Actual vs Predicted Rainfall (First 200 Test Days)')
plt.xlabel('Time Steps')
plt.ylabel('Rainfall (mm)')
plt.legend()
plt.grid(True)
plt.show()

### Interactive Weather Monitoring & Next-Day Rainfall Prediction

This cell creates an interactive dashboard using `ipywidgets`. Enter any non-negative rainfall measurements for the past 7 days and click **Predict Next-Day Rainfall** to run a prediction using our trained LSTM model. Use the **Reset** button to clear the input values.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

# Create 7 input widgets for the last 7 days of rainfall (mm)
inputs = [
    widgets.FloatText(value=0.0, description=f'Day {i} (mm):', min=0.0, step=0.1)
    for i in range(1, 8)
]

# Create buttons and output area
predict_btn = widgets.Button(description="Predict Next-Day Rainfall", button_style='primary')
reset_btn = widgets.Button(description="Reset", button_style='warning')
output = widgets.Output()

def handle_predict(b):
    with output:
        clear_output()
        try:
            # Gather and validate inputs
            raw_vals = []
            for i, inp in enumerate(inputs):
                v = inp.value
                if v is None or v < 0:
                    raise ValueError(f"Invalid input for Day {i+1}. Please enter a non-negative number.")
                raw_vals.append(v)

            # Convert list to numpy array
            raw_array = np.array(raw_vals).reshape(-1, 1)

            # Normalize using the fitted scaler
            normalized_vals = scaler.transform(raw_array)

            # Reshape for LSTM: [samples, time_steps, features] -> (1, 7, 1)
            lstm_input = normalized_vals.reshape((1, 7, 1))

            # Predict
            pred_normalized = model.predict(lstm_input, verbose=0)

            # Inverse transform to original scale (mm)
            pred_orig = scaler.inverse_transform(pred_normalized)[0][0]

            # Ensure no negative prediction is displayed
            pred_orig = max(0.0, pred_orig)

            # Display results
            print("\n=== Prediction Results ===")
            print(f"Previous 7-day rainfall: {', '.join(map(str, raw_vals))} mm")
            print(f"Predicted next-day (Day 8) rainfall: {pred_orig:.2f} mm")
            print("==========================")
            print("*Note: Predictions are generated solely by the trained LSTM model based on user input.")

        except Exception as e:
            print(f"Error: {str(e)}")

def handle_reset(b):
    with output:
        clear_output()
        for inp in inputs:
            inp.value = 0.0
        print("Inputs reset successfully.")

# Bind actions
predict_btn.on_click(handle_predict)
reset_btn.on_click(handle_reset)

# Render Layout
ui = widgets.VBox([
    widgets.Label(value="Enter previous 7 days of rainfall in millimeters (mm):"),
    widgets.GridBox(inputs, layout=widgets.Layout(grid_template_columns="repeat(3, 200px)")),
    widgets.HBox([predict_btn, reset_btn]),
    output
])

display(ui)

### Advanced Feature Engineering: Cyclical Seasonality & Rolling Windows

Below is the code to engineer cyclical time features (Month Sine/Cosine) and a 3-day rolling mean feature. Adding these features transforms our input from a univariate sequence into a multivariate sequence, allowing the LSTM to utilize both history and seasonal context.

In [ ]:
import numpy as np
import pandas as pd

# 1. Cyclical Month Encoding
# Since months are cyclic (12 wraps around to 1), we use sine and cosine transformations
daily['month_sin'] = np.sin(2 * np.pi * daily['month'] / 12.0)
daily['month_cos'] = np.cos(2 * np.pi * daily['month'] / 12.0)

# 2. Rolling Window Features
# Calculate a 3-day rolling mean within each state and district to smooth out sudden noise
daily['rolling_mean_3d'] = daily.groupby(['state', 'district'])['Rainfall_Normalized'].transform(lambda x: x.rolling(3, min_periods=1).mean())

print("Engineered Features Preview:")
display(daily[['state', 'district', 'date', 'Rainfall_Normalized', 'month_sin', 'month_cos', 'rolling_mean_3d']].head())


In [ ]:
def create_sequences_multivariate(df, feature_cols, target_col='Rainfall_Normalized', window_size=7):
    X, y = [], []
    # Group by state and district to maintain separate series boundaries
    grouped = df.groupby(['state', 'district'])

    for (state, district), group in grouped:
        # Ensure chronological order
        group_sorted = group.sort_values('date')

        # Extract all feature columns and the single target column
        feature_values = group_sorted[feature_cols].values
        target_values = group_sorted[target_col].values

        if len(feature_values) > window_size:
            for i in range(len(feature_values) - window_size):
                # Extract sequence of shape [window_size, num_features]
                X.append(feature_values[i : i + window_size])
                # The target is the step immediately following the window
                y.append(target_values[i + window_size])

    return np.array(X), np.array(y)

# Define multivariate features
features = ['Rainfall_Normalized', 'month_sin', 'month_cos', 'rolling_mean_3d']
window_size = 7

# Generate multivariate sequences
X_multi, y_multi = create_sequences_multivariate(daily, feature_cols=features, window_size=window_size)

print("Multivariate Input sequences shape (X_multi):", X_multi.shape)
print("Multivariate Target values shape (y_multi):", y_multi.shape)

In [ ]:
# 1. Chronological split of multivariate datasets
split_idx_multi = int(len(X_multi) * 0.8)

X_train_m, X_test_m = X_multi[:split_idx_multi], X_multi[split_idx_multi:]
y_train_m, y_test_m = y_multi[:split_idx_multi], y_multi[split_idx_multi:]

print(f"Multivariate Training set: {X_train_m.shape}, Test set: {X_test_m.shape}")

In [ ]:
# 2. Define multivariate LSTM architecture
model_multi = Sequential([
    LSTM(64, activation='relu', input_shape=(window_size, len(features)), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model_multi.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_multi.summary()

In [ ]:
# 3. Train the model using the engineered multivariate features
early_stop_m = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_multi = model_multi.fit(
    X_train_m, y_train_m,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop_m],
    verbose=1
)

In [ ]:
# 4. Evaluate the multivariate model predictions
y_pred_m = model_multi.predict(X_test_m)

# Inverse transform to original scale
y_test_m_orig = scaler.inverse_transform(y_test_m.reshape(-1, 1))
y_pred_m_orig = scaler.inverse_transform(y_pred_m.reshape(-1, 1))

# Calculate metrics
mae_m = mean_absolute_error(y_test_m_orig, y_pred_m_orig)
rmse_m = np.sqrt(mean_squared_error(y_test_m_orig, y_pred_m_orig))

print("===========================================================")
print("        LSTM MODEL PERFORMANCE COMPARISON (ORIGINAL SCALE)  ")
print("===========================================================")
print(f"Univariate LSTM (Pre-engineered):\n  - MAE:  {mae_orig:.4f} mm\n  - RMSE: {rmse_orig:.4f} mm")
print("-----------------------------------------------------------")
print(f"Multivariate LSTM (With Seasonality & Rolling Windows):\n  - MAE:  {mae_m:.4f} mm\n  - RMSE: {rmse_m:.4f} mm")
print("===========================================================")